# Visualizing ViT Attention

Loads a trained checkpoint and visualizes which patches the `[CLS]`
token attends to — this is the ViT equivalent of a CNN saliency map,
and it's the clearest way to confirm the model learned something
sensible rather than just memorizing the training set.

In [ ]:
import sys
sys.path.append("..")

import torch
import matplotlib.pyplot as plt

from src.vit import build_vit_from_config
from src.dataset import build_dataloaders, unnormalize, CIFAR10_CLASSES
from src.utils import load_config, load_checkpoint, get_device

config = load_config("../config/vit_config.yaml")
device = get_device(config["training"]["device"])

model = build_vit_from_config(config).to(device)
load_checkpoint(model, "../outputs/vit-run/best_model.pt", device=str(device))
model.eval()
print("Model loaded.")

In [ ]:
_, test_loader = build_dataloaders(config)
images, labels = next(iter(test_loader))
image, label = images[0:1].to(device), labels[0].item()

with torch.no_grad():
    logits, attn_maps = model(image, return_attn=True)

pred = logits.argmax(dim=-1).item()
print(f"True label: {CIFAR10_CLASSES[label]}  |  Predicted: {CIFAR10_CLASSES[pred]}")
print(f"Number of layers with attention maps: {len(attn_maps)}")

## Extract [CLS] -> patch attention from the last layer

For each attention head in the final block, take the attention row
for the `[CLS]` token (index 0) attending to all patch tokens
(indices 1 onward), then reshape it back into a 2D grid matching the
original patch layout.

In [ ]:
last_layer_attn = attn_maps[-1][0]   # (num_heads, seq_len, seq_len) for this one image
num_heads = last_layer_attn.shape[0]

patch_size = config["model"]["patch_size"]
grid_size = config["model"]["image_size"] // patch_size

fig, axes = plt.subplots(1, num_heads + 1, figsize=(4 * (num_heads + 1), 4))

orig_img = unnormalize(images[0]).permute(1, 2, 0).clamp(0, 1).numpy()
axes[0].imshow(orig_img)
axes[0].set_title(f"Original\n(true: {CIFAR10_CLASSES[label]}, pred: {CIFAR10_CLASSES[pred]})")
axes[0].axis("off")

for h in range(num_heads):
    cls_to_patches = last_layer_attn[h, 0, 1:].cpu()          # drop [CLS]->[CLS], keep [CLS]->patches
    attn_grid = cls_to_patches.reshape(grid_size, grid_size)
    axes[h + 1].imshow(attn_grid, cmap="viridis")
    axes[h + 1].set_title(f"Head {h} attention")
    axes[h + 1].axis("off")

plt.tight_layout()
plt.show()

## Exercise

- Run this on 5-10 different test images. Do different heads
  consistently focus on different regions (e.g., edges vs center)?
- Compare attention maps from an early layer (`attn_maps[0]`) vs the
  last layer (`attn_maps[-1]`) — attention in early layers tends to be
  more local/diffuse, later layers more semantic. Does that hold here?
- Try this on an image the model got WRONG. Does the attention map
  give any hint about why it misclassified it?